In [1]:
import jax.numpy as jnp
from jax import vmap
import numpy as np

class PiecewiseLinearPolynomialsHex:
    def __init__(self, N_t, N_x, N_y, t_min=0.0, t_max=1.0, x_min=-1.0, x_max=1.0, y_min=-1.0, y_max=1.0, rf_times=3, bdry_order=3, dg_type='Q1'):
        """
        Initialize a hexahedral mesh with custom ranges.

        Args:
            N_t: Number of divisions along the t-axis.
            N_x: Number of divisions along the x-axis.
            N_y: Number of divisions along the y-axis.
            t_min: Minimum value for t (default: 0.0).
            t_max: Maximum value for t (default: 1.0).
            x_min: Minimum value for x (default: -1.0).
            x_max: Maximum value for x (default: 1.0).
            y_min: Minimum value for y (default: -1.0).
            y_max: Maximum value for y (default: 1.0).
            rf_times: Refinement level for composite quadrature. 
            bdry_order: Order of boundary quadrature (number of points per face direction, default: 3).
        """
        self.N_t = N_t
        self.N_x = N_x
        self.N_y = N_y
        self.t_min = t_min
        self.t_max = t_max
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.rf_times = rf_times
        self.bdry_order = bdry_order
        self.N_elements = N_t * N_x * N_y  # Number of hexahedra
        self.partitions = self._generate_hexahedra()
        if dg_type == 'Q1':
            self.p_dof = 8
        elif dg_type == 'P1':
            self.p_dof = 4
        self.coeffs = jnp.zeros((self.N_elements, self.p_dof))
        self.quad_points, self.quad_weights, self.quad_element_indices, self.quad_points_parti, self.quad_weights_parti = self._setup_quadrature()
        # self.bdry_points_parti, self.bdry_weights_parti = self.get_boundary_quadrature()
        # self.normals = self.compute_normals()

    def _generate_hexahedra(self):
        """Generate the hexahedra for the mesh with custom ranges."""
        self.ht = (self.t_max - self.t_min) / self.N_t
        self.hx = (self.x_max - self.x_min) / self.N_x
        self.hy = (self.y_max - self.y_min) / self.N_y
        t_starts = jnp.linspace(self.t_min, self.t_max - self.ht, self.N_t)
        x_starts = jnp.linspace(self.x_min, self.x_max - self.hx, self.N_x)
        y_starts = jnp.linspace(self.y_min, self.y_max - self.hy, self.N_y)
        tl, xl, yl = jnp.meshgrid(t_starts, x_starts, y_starts, indexing='ij')
        # Define the 8 vertices for each hexahedron using binary offsets
        offsets_bin = jnp.array([[int(b) for b in f'{i:03b}'] for i in range(8)], dtype=jnp.float32)  # (8,3) binary 000 to 111
        deltas = jnp.array([self.ht, self.hx, self.hy])
        offsets = offsets_bin * deltas  # (8,3)
        lowers = jnp.stack([tl.ravel(), xl.ravel(), yl.ravel()], axis=1)[:, None, :]  # (N_elements, 1, 3)
        partitions = lowers + offsets[None, :, :]  # (N_elements, 8, 3)
        return partitions
        
    def _setup_quadrature(self):
        """
        Setup composite quadrature points and weights for hexahedral mesh.

        Returns:
            quad_points: Quadrature points, shape (N_quad, 3).
            quad_weights: Quadrature weights, shape (N_quad,).
            quad_element_indices: Indices of hexahedra, shape (N_quad,).
            quad_points_parti: Quadrature points per hexahedron, shape (N_elements, self.Nq, 3).
            quad_weights_parti: Quadrature weights per hexahedron, shape (N_elements, self.Nq).
        """
        # Composite quadrature with refinement level k = self.rf_times
        k = self.rf_times
        num_intervals = 2 ** k
        num_points = num_intervals + 1
        self.Nq = num_points ** 3

        def composite_quad(hex):
            t_low, x_low, y_low = hex[0]
            t_high, x_high, y_high = hex[-1]
            dt = (t_high - t_low) / num_intervals
            dx = (x_high - x_low) / num_intervals
            dy = (y_high - y_low) / num_intervals
            t_grid = jnp.linspace(t_low, t_high, num_points)
            x_grid = jnp.linspace(x_low, x_high, num_points)
            y_grid = jnp.linspace(y_low, y_high, num_points)
            tt, xx, yy = jnp.meshgrid(t_grid, x_grid, y_grid, indexing='ij')
            points = jnp.stack([tt.ravel(), xx.ravel(), yy.ravel()], axis=1)
            wt = jnp.full(num_points, dt)
            wt = wt.at[0].set(dt / 2)
            wt = wt.at[-1].set(dt / 2)
            wx = jnp.full(num_points, dx)
            wx = wx.at[0].set(dx / 2)
            wx = wx.at[-1].set(dx / 2)
            wy = jnp.full(num_points, dy)
            wy = wy.at[0].set(dy / 2)
            wy = wy.at[-1].set(dy / 2)
            weights_3d = wt[:, None, None] * wx[None, :, None] * wy[None, None, :]
            weights = weights_3d.ravel()
            return points, weights

        # Vectorize over hexahedra
        quad_points_parti, quad_weights_parti = vmap(composite_quad)(self.partitions)

        # Flatten for global quadrature
        quad_points = quad_points_parti.reshape(-1, 3)
        quad_weights = quad_weights_parti.reshape(-1)
        quad_element_indices = jnp.repeat(jnp.arange(self.N_elements), self.Nq)

        return quad_points, quad_weights, quad_element_indices, quad_points_parti, quad_weights_parti

In [2]:
import jax.numpy as jnp
from jax import vmap
import numpy as np
from scipy.integrate import tplquad
from scipy.integrate import dblquad, quad
from PwpHex_class import PiecewiseLinearPolynomialsHex


import math

# Define a complicated test function (non-polynomial to stress the quadrature)
def test_function(t, x, y):
    return jnp.sin(2 * jnp.pi * t) * jnp.cos(3 * jnp.pi * x) * jnp.exp(y) + t * x**2 * y**3 + jnp.log(1 + jnp.abs(t + x + y))

# Vectorized version for JAX quadrature points
def test_function_vec(points):
    t, x, y = points[:, 0], points[:, 1], points[:, 2]
    return test_function(t, x, y)

# SciPy version (uses NumPy)
def test_function_scipy(y, x, t):  # Order for tplquad: integrand(y, x, t) with dy dx dt
    return np.sin(2 * np.pi * t) * np.cos(3 * np.pi * x) * np.exp(y) + t * x**2 * y**3 + np.log(1 + np.abs(t + x + y))

# Function to compute quadrature integral
def compute_quadrature_integral(plph):
    return jnp.sum(test_function_vec(plph.quad_points) * plph.quad_weights)

# Function to compute SciPy integral (high accuracy)

def compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max):
    def g(t):
        def integrand(y, x):
            return test_function_scipy(y, x, t)
        integral, _ = dblquad(
            integrand,
            x_min, x_max,
            lambda x: y_min, lambda x: y_max,
            epsabs=1e-10, epsrel=1e-10
        )
        return integral
    total_integral, error = quad(
        g,
        t_min, t_max,
        epsabs=1e-6, epsrel=1e-6
    )
    return total_integral, error
    
# Basic check: sum of weights should equal domain volume
def check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max):
    computed_volume = jnp.sum(plph.quad_weights)
    exact_volume = (t_max - t_min) * (x_max - x_min) * (y_max - y_min)
    print(f"Computed volume: {computed_volume}, Exact volume: {exact_volume}")
    return abs(computed_volume - exact_volume) < 1e-10

# Test with specific parameters
N_t, N_x, N_y = 3, 3, 3  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 2.0
y_min, y_max = -1.0, 1.0
gauss_order = 5  # Default

plph1 = PiecewiseLinearPolynomialsHex1(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)
plph = PiecewiseLinearPolynomialsHex(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)

# Check volume
volume_check = check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max)
print(f"Volume check passed: {volume_check}")

# Compute integrals
quad_int = compute_quadrature_integral(plph)
quad_int1 = compute_quadrature_integral(plph1)
# scipy_int, scipy_error = compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max)

print(f"Gauss Quadrature integral: {quad_int}")
print(f"Composite Quadrature integral: {quad_int1}")

print(f"SciPy integral: {scipy_int} (estimated error: {scipy_error})")

# Compare (difference should be small, depending on gauss_order and mesh refinement)
difference = abs(scipy_int - quad_int1)
print(f"Absolute difference: {difference}")
print(f"Relative difference: {difference / abs(scipy_int) if quad_int != 0 else 'N/A'}")

# To further test correctness, you can increase gauss_order, N_t, N_x, N_y and observe convergence to SciPy value.2.9s

NameError: name 'PiecewiseLinearPolynomialsHex1' is not defined

In [4]:
4.31862038809356  - 4.318734645843506

-0.00011425774994577864

In [12]:
# Test with specific parameters
N_t, N_x, N_y = 3, 3, 3  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
gauss_order = 3  # Default

plph = PiecewiseLinearPolynomialsHex(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)

# Check volume
volume_check = check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max)
print(f"Volume check passed: {volume_check}")

# Compute integrals
quad_int = compute_quadrature_integral(plph)

Computed volume: 3.999999761581421, Exact volume: 4.0
Volume check passed: False


In [16]:
plph.quad_weights.shape

(3375,)

In [3]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu
from seq_minmax_opt_class import SeqMinMaxOptProb
from minmax_opt_class_mgnn_2d import MinMaxOptProb2D
from my_helpers import plot_comparison_seperate_combined_u_theta, compute_errors_combined_u_theta, plot_u_theta

In [4]:
def u0(x):
    return (-jnp.sign(x) + 1)*2

# Boundary at x=-1
def u1(x):
    return 4
vmap_u1 = vmap(u1, in_axes=0, out_axes=0)

# Boundary at x=2
def u2(x):
    return 0
vmap_u2 = vmap(u2, in_axes=0, out_axes=0)

def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x

def u_ref(t, x):
    return u0(x-2*t)

t_min = 0
t_max = 0.5
x_min = -0.5
x_max = 1.5


In [3]:
N_t = 8
N_x = 8
seq_trainer = SeqMinMaxOptProb(
    input_dim=2, num_segments=4,
    total_t_min=t_min, total_t_max=t_max, x_min=x_min, x_max=x_max,
    N_t=N_t, N_x=N_x, N_bc=2048, N_ic=4096,
    hidden_dims_list=[[64, 64, 64, 64, 64]], seed=0, pre_epochs=0,
    sigma=5, N_perturb=20000, k_h_rf_times=2, f_opt_kh=1,
    u0=u0, u1=vmap_u1, u2=vmap_u2, project="minmax_burgers_strong_moving_shock",
    gamma=0, N_test_gamma=1000, adjust_gamma=False, lambda_ic=10, lambda_bc=10, lambda_eqn=1,
    L1reg=0, sampling_method='uniform', notes='', 
    init_lrs=[1e-3, 1e-3, 1e-3], grade_epochs=[5000, 1000, 1000], decay_rates=[1e-3, 1e-3, 1e-3],
    res=res, loss_type='interior', k=0, f_plot=1000, partition='rect', dg_type='Q1', plt_err_dist=True, 
    set_wandb=False
)


In [2]:
import jax.numpy as jnp
from jax import vmap
import numpy as np

class PiecewiseLinearPolynomialsTetra:
    def __init__(self, N_t, N_x, N_y, t_min=0.0, t_max=1.0, x_min=-1.0, x_max=1.0, y_min=-1.0, y_max=1.0, quad_order=3, bdry_order=3, dg_type='P1'):
        """
        Initialize a tetrahedral mesh with custom ranges by dividing each hexahedron into 6 tetrahedrons.

        Args:
            N_t: Number of divisions along the t-axis.
            N_x: Number of divisions along the x-axis.
            N_y: Number of divisions along the y-axis.
            t_min: Minimum value for t (default: 0.0).
            t_max: Maximum value for t (default: 1.0).
            x_min: Minimum value for x (default: -1.0).
            x_max: Maximum value for x (default: 1.0).
            y_min: Minimum value for y (default: -1.0).
            y_max: Maximum value for y (default: 1.0).
            quad_order: Order of Gauss quadrature (1, 2, 3, or 4 for degree 1, 2, 3, or composite rule).
            bdry_order: Order of boundary quadrature (number of points per face direction, default: 3).
        """
        self.N_t = N_t
        self.N_x = N_x
        self.N_y = N_y
        self.t_min = t_min
        self.t_max = t_max
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.quad_order = quad_order
        self.bdry_order = bdry_order
        self.ht = (self.t_max - self.t_min) / self.N_t
        self.hx = (self.x_max - self.x_min) / self.N_x
        self.hy = (self.y_max - self.y_min) / self.N_y
        self.vol_per_tetra = self.ht * self.hx * self.hy / 6
        self.hex_partitions = self._generate_hexahedra()
        N_hex = N_t * N_x * N_y
        self.N_elements = N_hex * 6  # 6 tetrahedrons per hexahedron
        self.partitions = self._generate_tetrahedra()
        if dg_type == 'P1':
            self.p_dof = 4
        else:
            raise ValueError("Only 'P1' dg_type supported for tetrahedral elements.")
        self.coeffs = jnp.zeros((self.N_elements, self.p_dof))
        self.quad_points, self.quad_weights, self.element_indices, self.quad_points_parti, self.quad_weights_parti = self._setup_quadrature()
        # self.bdry_points_parti, self.bdry_weights_parti = self.get_boundary_quadrature()
        # self.normals = self.compute_normals()
        

    def _generate_hexahedra(self):
        """Generate the hexahedra for the mesh with custom ranges."""
        t_starts = jnp.linspace(self.t_min, self.t_max - self.ht, self.N_t)
        x_starts = jnp.linspace(self.x_min, self.x_max - self.hx, self.N_x)
        y_starts = jnp.linspace(self.y_min, self.y_max - self.hy, self.N_y)
        tl, xl, yl = jnp.meshgrid(t_starts, x_starts, y_starts, indexing='ij')
        # Define the 8 vertices for each hexahedron using binary offsets
        offsets_bin = jnp.array([[int(b) for b in f'{i:03b}'] for i in range(8)], dtype=jnp.float32)  # (8,3) binary 000 to 111
        deltas = jnp.array([self.ht, self.hx, self.hy])
        offsets = offsets_bin * deltas  # (8,3)
        lowers = jnp.stack([tl.ravel(), xl.ravel(), yl.ravel()], axis=1)[:, None, :]  # (N_elements, 1, 3)
        partitions = lowers + offsets[None, :, :]  # (N_elements, 8, 3)
        return partitions

    def _generate_tetrahedra(self):
        """Generate the tetrahedra by dividing each hexahedron into 6 tetrahedrons."""
        tetra_indices = jnp.array([[0,2,3,7], [0,3,1,7], [0,1,5,7], [0,5,4,7], [0,4,6,7], [0,6,2,7]])
        N_hex = self.N_t * self.N_x * self.N_y
        tiled_indices = jnp.repeat(tetra_indices[None, :, :], N_hex, axis=0)  # (N_hex, 6, 4)

        def select_tetra(hex_p, indices):
            return hex_p[indices]

        tet_part = vmap(select_tetra)(self.hex_partitions, tiled_indices)  # (N_hex, 6, 4, 3)
        return tet_part.reshape(self.N_elements, 4, 3)

    def _setup_quadrature(self):
        """
        Setup quadrature points and weights for tetrahedral mesh using barycentric coordinates.

        Returns:
            quad_points: Quadrature points, shape (N_quad, 3).
            quad_weights: Quadrature weights, shape (N_quad,).
            element_indices: Indices of tetrahedra, shape (N_quad,).
            quad_points_parti: Quadrature points per tetrahedron, shape (N_elements, Nq, 3).
            quad_weights_parti: Quadrature weights per tetrahedron, shape (N_elements, Nq).
        """
        if self.quad_order == 1:
            Nq = 1
            bary_points = jnp.array([[0.25, 0.25, 0.25, 0.25]])
            norm_weights = jnp.array([1.0])
        elif self.quad_order == 2:
            Nq = 4
            a = (1.0 + jnp.sqrt(1.8)) / 4.0
            b = (1.0 - a) / 3.0
            bary_points = jnp.array([
                [a, b, b, b],
                [b, a, b, b],
                [b, b, a, b],
                [b, b, b, a]
            ])
            norm_weights = jnp.array([0.25, 0.25, 0.25, 0.25])
        elif self.quad_order == 3:
            Nq = 8
            alpha = 0.0585540713266
            beta = 0.263112829241
            # Optimized weights to minimize moment errors for better accuracy
            w1_per = 0.07534533344104577
            w2_per = 0.17465466655895423
            def get_orbit(p):
                large = 1.0 - 3.0 * p
                small = p
                orbit = []
                for i in range(4):
                    bary = jnp.full(4, small)
                    bary = bary.at[i].set(large)
                    orbit.append(bary)
                return jnp.stack(orbit)
            orbit1 = get_orbit(alpha)
            orbit2 = get_orbit(beta)
            bary_points = jnp.vstack([orbit1, orbit2])
            norm_weights = jnp.array([w1_per] * 4 + [w2_per] * 4)
        elif self.quad_order == 4:
            Nq = 10
            bary_points = jnp.array([
                [1.0, 0.0, 0.0, 0.0],  # V0
                [0.0, 1.0, 0.0, 0.0],  # V1
                [0.0, 0.0, 1.0, 0.0],  # V2
                [0.0, 0.0, 0.0, 1.0],  # V3
                [0.5, 0.5, 0.0, 0.0],  # m_V0V1
                [0.0, 0.0, 0.5, 0.5],  # m_V2V3
                [0.5, 0.0, 0.5, 0.0],  # m_V0V2
                [0.5, 0.0, 0.0, 0.5],  # m_V0V3
                [0.0, 0.5, 0.5, 0.0],  # m_V1V2
                [0.0, 0.5, 0.0, 0.5],  # m_V1V3
            ])
            norm_weights = jnp.array([
                1.0/32, 1.0/32, 1.0/32, 1.0/32,  # vertices
                6.0/32, 6.0/32,  # poles
                4.0/32, 4.0/32, 4.0/32, 4.0/32   # equatorial
            ])
        else:
            raise ValueError("quad_order must be 1, 2, 3, or 4.")

        def map_to_tetra(tet):
            # tet shape (4, 3), bary_points (Nq, 4)
            points = jnp.einsum('qi,ij->qj', bary_points, tet)
            return points

        quad_points_parti = vmap(map_to_tetra)(self.partitions)  # (N_elements, Nq, 3)

        quad_weights_parti = jnp.full((self.N_elements, Nq), self.vol_per_tetra) * norm_weights[None, :]

        # Flatten for global quadrature
        quad_points = quad_points_parti.reshape(-1, 3)
        quad_weights = quad_weights_parti.reshape(-1)
        element_indices = jnp.repeat(jnp.arange(self.N_elements), Nq)

        return quad_points, quad_weights, element_indices, quad_points_parti, quad_weights_parti

In [9]:
import jax.numpy as jnp
from jax import vmap
import numpy as np
from scipy.integrate import tplquad
import math
import jax.numpy as jnp
from jax import vmap
import numpy as np
from scipy.integrate import tplquad
from scipy.integrate import dblquad, quad
from PwpHex_class import PiecewiseLinearPolynomialsHex

# Define a complicated test function (non-polynomial to stress the quadrature)
def test_function(t, x, y):
    return jnp.sin(2 * jnp.pi * t) * jnp.cos(3 * jnp.pi * x) * jnp.exp(y) + t * x**2 * y**3 + jnp.log(1 + jnp.abs(t + x + y))

# Vectorized version for JAX quadrature points
def test_function_vec(points):
    t, x, y = points[:, 0], points[:, 1], points[:, 2]
    return test_function(t, x, y)

# SciPy version (uses NumPy)
def test_function_scipy(y, x, t):  # Order for tplquad: integrand(y, x, t) with dy dx dt
    return np.sin(2 * np.pi * t) * np.cos(3 * np.pi * x) * np.exp(y) + t * x**2 * y**3 + np.log(1 + np.abs(t + x + y))

# Function to compute quadrature integral
def compute_quadrature_integral(plph):
    return jnp.sum(test_function_vec(plph.quad_points) * plph.quad_weights)

# Function to compute SciPy integral (high accuracy)
def compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max):
    integral, error = tplquad(
        test_function_scipy,
        t_min, t_max,  # dt
        lambda t: x_min, lambda t: x_max,  # dx
        lambda t, x: y_min, lambda t, x: y_max,  # dy
        epsabs=1e-7, epsrel=1e-7
    )
    return integral, error

# Basic check: sum of weights should equal domain volume
def check_volume(plph, t_min, t_max, x_min, x_max, y_min, y_max):
    computed_volume = jnp.sum(plph.quad_weights)
    exact_volume = (t_max - t_min) * (x_max - x_min) * (y_max - y_min)
    print(f"Computed volume: {computed_volume}, Exact volume: {exact_volume}")
    return abs(computed_volume - exact_volume) < 1e-7

# Test with specific parameters
N_t, N_x, N_y = 40, 40, 40  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
quad_order = 4  # Default

plphTetra = PiecewiseLinearPolynomialsTetra(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, quad_order)

plpHex = PiecewiseLinearPolynomialsHex(50, 50, 50, t_min, t_max, x_min, x_max, y_min, y_max, 3)

# Check volume
volume_check = check_volume(plphTetra, t_min, t_max, x_min, x_max, y_min, y_max)
print(f"Volume check passed: {volume_check}")

# Compute integrals
quad_int = compute_quadrature_integral(plphTetra)
quad_ref = compute_quadrature_integral(plpHex)

# scipy_int, scipy_error = compute_scipy_integral(t_min, t_max, x_min, x_max, y_min, y_max)

print(f"Quadrature integral: {quad_int}")
print(f"Reference Quadrature integral: {quad_ref}")

# print(f"SciPy integral: {scipy_int} (estimated error: {scipy_error})")

# Compare (difference should be small, depending on gauss_order and mesh refinement)
difference = abs(quad_int - quad_ref)
print(f"Absolute difference: {difference}")
print(f"Relative difference: {difference / abs(quad_ref) if quad_ref != 0 else 'N/A'}")

# To further test correctness, you can increase gauss_order, N_t, N_x, N_y and observe convergence to SciPy value.2.9s

Computed volume: 4.000000476837158, Exact volume: 4.0
Volume check passed: False
Quadrature integral: 2.1815805435180664
Reference Quadrature integral: 2.181222915649414
Absolute difference: 0.00035762786865234375
Relative difference: 0.00016395750571973622


In [14]:
offsets_bin = jnp.array([[int(b) for b in f'{i:02b}'] for i in range(4)], dtype=jnp.float32)  # (8,3) binary 000 to 111


In [8]:
N_t, N_x, N_y = 3, 3, 3  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
gauss_order = 5  # Default

pwp = PiecewiseLinearPolynomialsHex(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, gauss_order)

In [12]:
partitions = pwp.partitions[pwp.element_indices]
t_low = partitions[:, 0, 0]
x_low = partitions[:, 0, 1]
y_low = partitions[:, 0, 2]
t_high = partitions[:, -1, 0]
x_high = partitions[:, -1, 1]
y_high = partitions[:, -1, 2]
d_t = (t_high - t_low)
d_x = (x_high - x_low)
d_y = (y_high - y_low)

In [5]:
a = (1.0 + jnp.sqrt(1.8)) / 4.0
b = (1.0 - a) / 3.0

In [6]:
b

Array(0.1381966, dtype=float32, weak_type=True)

In [8]:
N_t, N_x, N_y = 10, 10, 10  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
quad_order = 3  # Default

plphTetra = PiecewiseLinearPolynomialsTetra(N_t, N_x, N_y, t_min, t_max, x_min, x_max, y_min, y_max, quad_order)

In [9]:
plphTetra.N_elements

6000

In [11]:
plphTetra.quad_points_parti.shape

(6000, 8, 3)

In [12]:
plphTetra.quad_points.shape

(48000, 3)

In [13]:
plpHex = PiecewiseLinearPolynomialsHex(20, 20, 20, t_min, t_max, x_min, x_max, y_min, y_max, 5)
plpHex.quad_points_parti.shape

(8000, 125, 3)

In [27]:
current_key = jax.random.PRNGKey(0)

In [28]:
current_key

Array([0, 0], dtype=uint32)

In [29]:
key, current_key = jax.random.split(current_key)

In [30]:
current_key

Array([ 928981903, 3453687069], dtype=uint32)

In [3]:
jnp.array([[int(b) for b in f'{i:03b}'] for i in range(8)], dtype=jnp.float32)

Array([[0., 0., 0.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 1.],
       [1., 0., 0.],
       [1., 0., 1.],
       [1., 1., 0.],
       [1., 1., 1.]], dtype=float32)

In [5]:
jnp.array([0, 0, 1, 2]).astype(jnp.int32)

Array([0, 0, 1, 2], dtype=int32)

In [2]:
import jax.numpy as jnp
from jax import vmap
import numpy as np
from scipy.integrate import tplquad
from scipy.integrate import dblquad, quad
from PwpHex_class import PiecewiseLinearPolynomialsHex
from PwpRect_class import PiecewiseLinearPolynomialsRect
import jax

In [7]:
# Test with specific parameters
N_t, N_x, N_y = 40, 40, 40  # Small number of divisions for testing
t_min, t_max = 0.0, 1.0
x_min, x_max = -1.0, 1.0
y_min, y_max = -1.0, 1.0
quad_order = 4  # Default

plpHex = PiecewiseLinearPolynomialsHex(50, 50, 50, t_min, t_max, x_min, x_max, y_min, y_max, 2, bdry_rf_times=3)
plpRect = PiecewiseLinearPolynomialsRect(50, 50, t_min, t_max, x_min, x_max, 2, bdry_rf_times=3)

In [3]:
plpHex

(125000, 6, 81, 3)

In [4]:
plpHex.quad_points_parti.shape

(125000, 125, 3)

In [13]:
import jax.numpy as jnp
from jax import vmap

# Assuming the PiecewiseLinearPolynomialsHex class is defined as provided in the conversation history

def F(p):
    """Define a complicated 3D vector field F(x, y, z) = (x^2 * y + sin(z) + exp(x*y), 
    y^2 * z + cos(x) + sin(y*z), z^2 * x + exp(y) + cos(x*z))"""
    x, y, z = p
    Fx = 10*(x**2 * y + jnp.sin(z) + jnp.exp(x * y))
    Fy = 10*(y**2 * z + jnp.cos(x) + jnp.sin(y * z))
    Fz = 10*(z**2 * x + jnp.exp(y) + jnp.cos(x * z))
    return jnp.array([Fx, Fy, Fz])

def div_F(p):
    """Analytic divergence of the vector field: div(F) = dFx/dx + dFy/dy + dFz/dz"""
    # Use automatic differentiation for accuracy, but could compute analytically
    div_F = jnp.trace(jax.jacobian(F)(p))
    return div_F

# Function to compute volume integral of div(F) for each element
def compute_volume_integrals(mesh):
    # Vectorize div_F over quadrature points per element
    div_F_vmap = vmap(div_F)
    # Compute integral per element: sum(weights * div_F(points))
    def vol_int_elem(e):
        return jnp.sum(div_F_vmap(mesh.quad_points_parti[e]) * mesh.quad_weights_parti[e])
    return vmap(vol_int_elem)(jnp.arange(mesh.N_elements))

# Function to compute surface flux integral of F · n for each element
def compute_surface_integrals(mesh):
    # Vectorize F over points
    F_vmap = vmap(F)
    
    # Compute flux for a single face of a single element
    def face_flux(e, f):
        points = mesh.bdry_points_parti[e, f]
        weights = mesh.bdry_weights_parti[e, f]
        normal = mesh.normals[e, f]
        F_vals = F_vmap(points)
        dots = jnp.einsum('nd,d->n', F_vals, normal)  # F · n for each point
        return jnp.sum(dots * weights)
    
    # Sum over all 6 faces for a single element
    def elem_flux(e):
        return jnp.sum(vmap(lambda f: face_flux(e, f))(jnp.arange(6)))
    
    # Vectorize over elements
    return vmap(elem_flux)(jnp.arange(mesh.N_elements))

# Test function
def test_divergence_theorem(mesh, tol=1e-6):
    vol_integrals = compute_volume_integrals(mesh)
    surf_integrals = compute_surface_integrals(mesh)
    differences = jnp.abs(vol_integrals - surf_integrals)
    all_close = jnp.all(differences < tol)
    print("Volume integrals:", vol_integrals)
    print("Surface integrals:", surf_integrals)
    print("Differences:", differences)
    if not all_close:
        print("Max differences", np.max(differences))
    print("All within tolerance?", all_close)
    return all_close

# Example usage
# Create a mesh instance (adjust parameters as needed; higher rf_times for better accuracy)
mesh = PiecewiseLinearPolynomialsHex(N_t=5, N_x=5, N_y=5, rf_times=6, bdry_rf_times=6)  # Use rf_times=4 for higher refinement

# Run the test
test_divergence_theorem(mesh, tol=1e-5)  # Adjust tol based on refinement level

Volume integrals: [-1.2712920e-01 -2.2625580e-01 -2.8740630e-01 -3.4855682e-01
 -4.4768330e-01 -2.3233116e-01 -1.9550531e-01 -1.4821979e-01
 -1.0093427e-01 -6.4108431e-02 -3.0220920e-01 -1.5129693e-01
  4.2692036e-04  1.5215074e-01  3.0306306e-01 -3.3438960e-01
 -9.2763752e-02  1.5932181e-01  4.1140738e-01  6.5303326e-01
 -3.2960436e-01 -1.9130995e-02  3.2931855e-01  6.7776811e-01
  9.8824137e-01 -2.7623546e-01 -3.3416566e-01 -3.5431525e-01
 -3.7446481e-01 -4.3239492e-01 -3.5560343e-01 -2.7758119e-01
 -1.8929471e-01 -1.0100827e-01 -2.2986010e-02 -3.8355234e-01
 -1.9144359e-01  1.2811526e-03  1.9400589e-01  3.8611457e-01
 -3.5320866e-01 -7.0386395e-02  2.2270009e-01  5.1578653e-01
  7.9860884e-01 -2.5988182e-01  9.1788009e-02  4.8123848e-01
  8.7068886e-01  1.2223587e+00 -4.1165936e-01 -4.3746173e-01
 -4.2664644e-01 -4.1583115e-01 -4.4163358e-01 -4.6065810e-01
 -3.5050803e-01 -2.3125674e-01 -1.1200541e-01 -1.8553623e-03
 -4.4578952e-01 -2.2155301e-01  2.1366156e-03  2.2582625e-01
  4.50

Array(True, dtype=bool)

In [13]:
import jax.numpy as jnp
from jax import vmap

# Assuming the PiecewiseLinearPolynomialsRect class is defined as provided

# Define a complicated 2D vector field F(t, x)
def F(point):
    t, x = point
    return jnp.array([
        jnp.sin(t) * x**2 + jnp.cos(2 * t) * jnp.exp(x),
        jnp.cos(x) * t**3 + jnp.sin(3 * x) * jnp.log(1 + jnp.abs(t))
    ])

# Define the divergence div(F) analytically for accuracy
def div_F(point):
    t, x = point
    # ∂/∂t of first component + ∂/∂x of second component
    return (
        jnp.cos(t) * x**2 - 2 * jnp.sin(2 * t) * jnp.exp(x) +  # ∂F1/∂t
        -jnp.sin(x) * t**3 + 3 * jnp.cos(3 * x) * jnp.log(1 + jnp.abs(t))  # ∂F2/∂x
    )

# Function to compute area integral of div(F) for each element
def compute_area_integrals(mesh):
    # Vectorize div_F over quadrature points per element
    div_F_vmap = vmap(div_F)
    # Compute integral per element: sum(weights * div_F(points))
    def area_int_elem(e):
        return jnp.sum(div_F_vmap(mesh.quad_points_parti[e]) * mesh.quad_weights_parti[e])
    return vmap(area_int_elem)(jnp.arange(mesh.N_elements))

# Function to compute boundary flux integral of F · n for each element
def compute_boundary_integrals(mesh):
    # Vectorize F over points
    F_vmap = vmap(F)
    
    # Compute flux for a single edge of a single element
    def edge_flux(e, f):
        points = mesh.bdry_points_parti[e, f]
        weights = mesh.bdry_weights_parti[e, f]
        normal = mesh.normals[e, f]
        F_vals = F_vmap(points)
        dots = jnp.einsum('nd,d->n', F_vals, normal)  # F · n for each point
        return jnp.sum(dots * weights)
    
    # Sum over all 4 edges for a single element
    def elem_flux(e):
        return jnp.sum(vmap(lambda f: edge_flux(e, f))(jnp.arange(4)))
    
    # Vectorize over elements
    return vmap(elem_flux)(jnp.arange(mesh.N_elements))

# Test function
def test_divergence_theorem(mesh, tol=1e-6):
    area_integrals = compute_area_integrals(mesh)
    boundary_integrals = compute_boundary_integrals(mesh)
    differences = jnp.abs(area_integrals - boundary_integrals)
    all_close = jnp.all(differences < tol)
    print("Area integrals:", area_integrals)
    print("Boundary integrals:", boundary_integrals)
    print("Differences:", differences)
    print("All within tolerance?", all_close)
    return all_close

# Example usage
# Create a mesh instance (adjust parameters as needed; higher rf_times for better accuracy)
mesh = PiecewiseLinearPolynomialsRect(N_t=5, N_x=5, rf_times=5, bdry_rf_times=5)  # Use rf_times=4 for higher refinement

# Run the test
test_divergence_theorem(mesh, tol=1e-5)  # Adjust tol based on refinement level

Area integrals: [ 2.2111109e-02 -3.2557134e-02 -7.4686319e-02 -9.9248916e-02
 -1.0231157e-01  2.1738745e-04 -2.5013108e-02 -4.0954288e-02
 -4.0892910e-02 -2.0513440e-02 -9.5133949e-03 -3.0295569e-02
 -4.2298056e-02 -3.7151668e-02 -1.0303972e-02 -2.6019299e-02
 -1.0108594e-01 -1.5960243e-01 -1.9208175e-01 -1.9419149e-01
 -3.4576751e-02 -1.9644326e-01 -3.2864568e-01 -4.1920844e-01
 -4.6321097e-01]
Boundary integrals: [ 2.2109251e-02 -3.2562550e-02 -7.4694768e-02 -9.9259824e-02
 -1.0232434e-01  2.1801703e-04 -2.5011294e-02 -4.0951502e-02
 -4.0889017e-02 -2.0508312e-02 -9.5112864e-03 -3.0289821e-02
 -4.2289130e-02 -3.7139669e-02 -1.0289058e-02 -2.6019040e-02
 -1.0108508e-01 -1.5960118e-01 -1.9207986e-01 -1.9418865e-01
 -3.4579318e-02 -1.9645080e-01 -3.2865742e-01 -4.1922349e-01
 -4.6322840e-01]
Differences: [1.8570572e-06 5.4165721e-06 8.4489584e-06 1.0907650e-05 1.2770295e-05
 6.2957406e-07 1.8142164e-06 2.7865171e-06 3.8929284e-06 5.1278621e-06
 2.1085143e-06 5.7481229e-06 8.9257956e-06 

Array(False, dtype=bool)

In [14]:
mesh.bdry_points_parti.shape

(25, 4, 33, 2)